# Notebook 01 — Exploratory Data Analysis

**Goal:** Understand the structure, distributions, and patterns in the Chicago Crime Dataset before modelling.

Run `src/data/download.py` and `src/data/clean.py` first to populate `data/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

os.makedirs('../outputs/figures', exist_ok=True)

## 1. Load cleaned data

In [ ]:
df = pd.read_csv('../data/samples/chicago_sample.csv', low_memory=False)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
df.info()
df.describe(include='all').T

## 2. Missing value analysis

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print('Missing values per column:')
print(missing)

## 3. Target variable — Arrest class distribution

In [ ]:
counts = df['Arrest'].value_counts()
labels = ['No Arrest (0)', 'Arrest (1)']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(labels, counts.values, color=['#4878cf', '#d65f5f'])
axes[0].set_title('Arrest Count')
axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%',
            colors=['#4878cf', '#d65f5f'], startangle=90)
axes[1].set_title('Arrest Rate')

plt.suptitle('Class Distribution — Arrest', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/arrest_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Arrest rate: {df['Arrest'].mean():.1%}")

## 4. Crime type distribution

In [ ]:
top_crimes = df['Primary Type'].value_counts().head(15)

plt.figure(figsize=(10, 6))
top_crimes.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 15 Crime Types', fontsize=13)
plt.xlabel('Number of Incidents')
plt.tight_layout()
plt.savefig('../outputs/figures/crime_type_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Temporal patterns

In [ ]:
df['Hour']      = df['Date'].dt.hour
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Month']     = df['Date'].dt.month
df['Year']      = df['Date'].dt.year

day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Hour of day
hour_counts = df.groupby('Hour').size()
axes[0,0].plot(hour_counts.index, hour_counts.values, 'o-', color='steelblue')
axes[0,0].set_title('Crimes by Hour of Day')
axes[0,0].set_xlabel('Hour')
axes[0,0].set_ylabel('Count')

# Day of week
dow_counts = df.groupby('DayOfWeek').size()
axes[0,1].bar(day_names, dow_counts.values, color='coral')
axes[0,1].set_title('Crimes by Day of Week')
axes[0,1].set_xlabel('Day')

# Month
month_counts = df.groupby('Month').size()
axes[1,0].bar(month_counts.index, month_counts.values, color='mediumseagreen')
axes[1,0].set_title('Crimes by Month')
axes[1,0].set_xlabel('Month')
axes[1,0].set_xticks(range(1,13))

# Year trend
year_counts = df.groupby('Year').size()
axes[1,1].plot(year_counts.index, year_counts.values, 'o-', color='purple')
axes[1,1].set_title('Crimes by Year')
axes[1,1].set_xlabel('Year')

plt.suptitle('Temporal Crime Patterns', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/figures/temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Arrest rate by hour
arrest_by_hour = df.groupby('Hour')['Arrest'].mean()

plt.figure(figsize=(10, 4))
arrest_by_hour.plot(kind='line', marker='o', color='crimson')
plt.title('Arrest Rate by Hour of Day', fontsize=13)
plt.xlabel('Hour')
plt.ylabel('Arrest Rate')
plt.axhline(df['Arrest'].mean(), ls='--', color='grey', label='Overall mean')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/arrest_rate_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Geographic distribution

In [ ]:
df_geo = df.dropna(subset=['Latitude', 'Longitude'])

plt.figure(figsize=(9, 11))
plt.scatter(df_geo['Longitude'], df_geo['Latitude'],
            alpha=0.01, s=0.4, c='steelblue')
plt.title('Chicago Crime Locations', fontsize=13)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.savefig('../outputs/figures/geo_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Arrest rate by crime type

In [ ]:
arrest_by_type = df.groupby('Primary Type')['Arrest'].mean().sort_values()

plt.figure(figsize=(10, 8))
arrest_by_type.plot(kind='barh', color='darkorange')
plt.axvline(df['Arrest'].mean(), ls='--', color='red', label='Overall mean')
plt.title('Arrest Rate by Crime Type', fontsize=13)
plt.xlabel('Arrest Rate')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/arrest_by_crime_type.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. District-level analysis

In [ ]:
district_stats = df.groupby('District').agg(
    crime_count=('Arrest', 'count'),
    arrest_rate=('Arrest', 'mean')
).sort_values('crime_count', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

district_stats['crime_count'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Total Crimes by District')
axes[0].set_xlabel('District')

district_stats['arrest_rate'].sort_values().plot(kind='barh', ax=axes[1], color='coral')
axes[1].axvline(df['Arrest'].mean(), ls='--', color='red', label='Overall mean')
axes[1].set_title('Arrest Rate by District')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/district_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Correlation heatmap

In [ ]:
from src.data.features import engineer_features

df_feat = engineer_features(df.copy())
numeric_cols = ['Hour', 'DayOfWeek', 'Month', 'Year',
                'IsWeekend', 'IsNight', 'PrimaryType_enc',
                'LocationDesc_enc', 'Season_enc', 'Domestic_enc',
                'District', 'Community Area', 'Arrest']
numeric_cols = [c for c in numeric_cols if c in df_feat.columns]

corr = df_feat[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Domestic vs non-domestic crimes

In [ ]:
if 'Domestic' in df.columns:
    domestic_arrest = df.groupby('Domestic')['Arrest'].mean()
    domestic_arrest.index = ['Non-Domestic', 'Domestic']
    domestic_arrest.plot(kind='bar', color=['#4878cf', '#d65f5f'], figsize=(6, 4))
    plt.title('Arrest Rate: Domestic vs Non-Domestic')
    plt.ylabel('Arrest Rate')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig('../outputs/figures/domestic_arrest_rate.png', dpi=150, bbox_inches='tight')
    plt.show()